# Loan Default Prediction — Feature Engineering

**Objective:** Transform raw applicant data into model-ready features, based on patterns identified in EDA.

**What we'll do:**
- Clean and impute missing values
- Engineer domain-relevant ratio features
- Handle anomalous values
- Encode categorical variables
- Save the processed dataset for modelling

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv('../data/application_train.csv')
print(f'Shape: {df.shape}')

## 2. Drop High-Missingness Columns

Columns with >60% missing values are unlikely to contribute signal and will cause noise during imputation.

In [ ]:
MISSING_THRESHOLD = 0.60

missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()

print(f'Dropping {len(cols_to_drop)} columns with >{MISSING_THRESHOLD*100:.0f}% missing:')
for c in cols_to_drop:
    print(f'  {c}: {missing_pct[c]:.1%}')

df = df.drop(columns=cols_to_drop)
print(f'\nShape after drop: {df.shape}')

## 3. Fix Anomalous Values

In [ ]:
# DAYS_EMPLOYED = 365243 is a sentinel for 'not employed / pensioner'
df['EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# XNA in CODE_GENDER — treat as missing
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', np.nan)

print('DAYS_EMPLOYED anomaly flagged:', df['EMPLOYED_ANOMALY'].sum())
print('CODE_GENDER XNA replaced:', df['CODE_GENDER'].isna().sum())

## 4. Age & Employment Features

In [ ]:
# Convert negative day values to positive years
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
df['EMPLOYMENT_YEARS'] = -df['DAYS_EMPLOYED'] / 365
df['REGISTRATION_YEARS'] = -df['DAYS_REGISTRATION'] / 365
df['ID_PUBLISH_YEARS'] = -df['DAYS_ID_PUBLISH'] / 365

# Age group buckets
df['AGE_GROUP'] = pd.cut(
    df['AGE_YEARS'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['<25', '25-35', '35-45', '45-55', '55+']
)

# Default rate by age group
print('Default rate by age group:')
print(df.groupby('AGE_GROUP')['TARGET'].mean().round(3))

## 5. Financial Ratio Features

Domain knowledge: a borrower's ability to repay is better captured by ratios than raw amounts.

In [ ]:
# Credit burden
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

# Goods vs credit
df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']

# Income per family member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, 1)

# Employment stability
df['EMPLOYMENT_TO_AGE'] = df['EMPLOYMENT_YEARS'] / df['AGE_YEARS']

ratio_features = [
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
    'GOODS_CREDIT_RATIO', 'INCOME_PER_PERSON', 'EMPLOYMENT_TO_AGE'
]

print('New ratio features — correlation with TARGET:')
df[ratio_features + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values()

## 6. EXT_SOURCE Aggregate Features

The external credit scores are among the strongest predictors. Combining them captures information not present in any single score.

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

df['EXT_SOURCE_MEAN'] = df[ext_cols].mean(axis=1)
df['EXT_SOURCE_MIN']  = df[ext_cols].min(axis=1)
df['EXT_SOURCE_MAX']  = df[ext_cols].max(axis=1)
df['EXT_SOURCE_STD']  = df[ext_cols].std(axis=1)
df['EXT_SOURCE_PROD'] = df[ext_cols].prod(axis=1)

ext_features = ['EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_MAX',
                'EXT_SOURCE_STD', 'EXT_SOURCE_PROD']

print('EXT_SOURCE aggregate features — correlation with TARGET:')
df[ext_features + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values()

## 7. Document & Social Flag Features

In [ ]:
# Count how many documents the applicant provided
doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)

# Count social circle observations
social_cols = [c for c in df.columns if 'SOCIAL_CIRCLE' in c]
if social_cols:
    df['SOCIAL_CIRCLE_DEFAULT_MEAN'] = df[[c for c in social_cols if 'DEF' in c]].mean(axis=1)

print(f'Document flag columns: {len(doc_cols)}')
print(f'DOCUMENT_COUNT correlation with TARGET: {df["DOCUMENT_COUNT"].corr(df["TARGET"]):.4f}')

## 8. Encode Categorical Variables

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {len(cat_cols)}')
for c in cat_cols:
    print(f'  {c}: {df[c].nunique()} unique values')

In [ ]:
# Binary encode columns with 2 unique values, one-hot encode the rest
binary_cols = [c for c in cat_cols if df[c].nunique() <= 2]
multi_cols  = [c for c in cat_cols if df[c].nunique() > 2]

# Binary encoding
for col in binary_cols:
    df[col] = pd.factorize(df[col])[0]

# One-hot encoding
df = pd.get_dummies(df, columns=multi_cols, drop_first=False, dtype=int)

# Drop AGE_GROUP (was only used for analysis)
df = df.drop(columns=['AGE_GROUP'], errors='ignore')

print(f'Shape after encoding: {df.shape}')

## 9. Impute Remaining Missing Values

In [ ]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f'Columns still with missing values: {len(remaining_missing)}')

# Median imputation for numeric columns
for col in remaining_missing.index:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(f'Remaining nulls after imputation: {df.isnull().sum().sum()}')

## 10. Feature Summary

In [ ]:
# Top 20 features by absolute correlation with TARGET
feature_corr = df.corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top20 = feature_corr.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='#1565C0')
ax.set_xlabel('|Correlation with TARGET|')
ax.set_title('Top 20 Features by Absolute Correlation with Default', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 20 features:')
print(top20)

## 11. Save Processed Dataset

In [ ]:
df.to_csv('../data/application_train_processed.csv', index=False)

print(f'Saved processed dataset.')
print(f'Final shape: {df.shape}')
print(f'Target distribution: {df["TARGET"].value_counts(normalize=True).round(3).to_dict()}')

## Summary

| Step | Detail |
|---|---|
| Dropped high-missing columns | Removed columns with >60% missing |
| Fixed anomalies | `DAYS_EMPLOYED` sentinel → binary flag + NaN |
| Age & time features | Converted days to years; binned age groups |
| Financial ratios | Credit/income, annuity/income, goods/credit, income per person |
| EXT_SOURCE aggregates | Mean, min, max, std, product across 3 scores |
| Document count | Summed all FLAG_DOCUMENT columns |
| Categorical encoding | Binary encode (2-class) + one-hot encode (multi-class) |
| Imputation | Median imputation for remaining numeric NaNs |

**Next:** Modelling in `03_modeling.ipynb`